# optionm'

In [ ]:
open rust
open rust_operators

In [ ]:
///- --test static

open testing

## optionm'

### default_value

In [ ]:
inl default_value d =
    optionm.defaultWith d

In [1]:
///- --test

None
|> default_value 3i32
|> _assert_eq 3i32

__assert_eq { expected = 3 }

### (/??)

In [ ]:
inl (/??) a b =
    a |> default_value b

In [2]:
///- --test

None /?? 3i32
|> _assert_eq 3i32

__assert_eq { expected = 3 }

### default_with

In [ ]:
inl default_with fn = function
    | Some x => x
    | None => fn ()

In [3]:
///- --test

None
|> default_with fun () => 3i32
|> _assert_eq 3i32

__assert_eq { expected = 3 }

### choose

In [ ]:
inl choose fn a b =
    match a, b with
    | Some x, Some y => fn x y |> Some
    | _ => None

In [4]:
///- --test

(Some 2i32, Some 3)
||> choose (+)
|> _assert_eq (Some 5)

__assert_eq { expected = Some(5) }

### iter

In [ ]:
inl iter fn = function
    | Some x => fn x
    | None => ()

In [5]:
///- --test

inl n = mut 1i32
inl fn =
    fun n' =>
        n <- *n + n'
Some 1i32 |> iter fn
None |> iter fn
*n
|> _assert_eq 2i32

__assert_eq { expected = 2 }

### flatten

In [ ]:
inl flatten x =
    match x with
    | Some (Some x) => Some x
    | _ => None

## fsharp

### option'

In [ ]:
inl option'_types () : () =
    backend_switch {
        Gleam = fun () => global "import gleam/option"
    }
nominal option'_gleam t =
    `(
        option'_types ()
        $'' : $'option.Option(`t)'
    )
type option'_switch t =
    {
        Gleam : option'_gleam t
        Lua : t
        Fsharp : $"`t option"
        Python : t
        Rust : $'Option<`t>'
        TypeScript : t
        Zig : $'?`(t)'
        Lean : $'(Option `t)'
    }
nominal option' t = $'backend_switch `(option'_switch t)'

### none'

In [ ]:
inl none' forall t. () : option' t =
    backend_switch {
        Gleam = fun () => $'option.None' : option' t
        Lua = fun () => $'nil' : option' t
        Fsharp = fun () => $'None' : option' t
        Python = fun () => $'None' : option' t
        Rust = fun () => $'None' : option' t
        TypeScript = fun () => $'undefined' : option' t
        Zig = fun () => $'null' : option' t
        Lean = fun () => $'none' : option' t
    }

### some'

In [ ]:
inl some' forall t. (x : t) : option' t =
    backend_switch {
        Gleam = fun () => $'!x |> option.Some' : option' t
        Lua = fun () => $'!x ' : option' t
        Fsharp = fun () => $'Some !x ' : option' t
        Python = fun () => $'!x \# some\' ' : option' t
        Rust = fun () => $'Some(!x.clone())' : option' t
        TypeScript = fun () => $'!x ' : option' t
        Zig = fun () => $'!x ' : option' t
        Lean = fun () => $'(some !x)' : option' t
    }

### default_value'

In [ ]:
inl default_value' forall t. (value : t) (x : option' t) : t =
    backend_switch {
        Gleam = fun () => $'!x |> option.unwrap(!value)' : t
        Lua = fun () => $'(function() if !x ~= nil then return !x else return !value end end)()' : t
        Fsharp = fun () => $'!x |> Option.defaultValue !value ' : t
        Python = fun () => $'(!value if !x is None else !x)' : t
        Rust = fun () => $'!x.unwrap_or(!value)' : t
        TypeScript = fun () => $'(!x === undefined ? !value : !x)' : t
        Zig = fun () => $'(!x orelse !value)' : t
        Lean = fun () => $'((!x).getD !value)' : t
    }

### get'

In [ ]:
inl get' forall t. (x : option' t) : t =
    backend_switch {
        Gleam = fun () =>
            option'_types ()
            $'!x |> option.unwrap (Nil)' : t
        Lua = fun () => $'!x ' : t
        Fsharp = fun () => $'!x |> Option.get' : t
        Python = fun () => $'!x ' : t
        Rust = fun () => $'!x.unwrap()' : t
        TypeScript = fun () => $'!x ' : t
        Zig = fun () => $'!x.?' : t
        Lean = fun () => $'(!x).get\!' : t
    }

### box

In [ ]:
inl box forall t. (x : option t) : option' t =
    match x with
    | Some x => some' x
    | None => none' ()

### map

In [ ]:
inl map forall t u. (fn : t -> u) (x : option' t) : option' u =
    backend_switch {
        Gleam = fun () =>
            option'_types ()
            $'!x |> option.map(!fn)' : option' u
        Lua = fun () =>
            if x =. none' ()
            then none' ()
            else fn $'!x ' |> fun x => $'!x ' : option' u
        Fsharp = fun () =>
            inl fn = join fn
            $'!x |> Option.map !fn ' : option' u
        Python = fun () =>
            if x =. none' ()
            then none' ()
            else fn $'!x ' |> fun x => $'!x ' : option' u
        Rust = fun () =>
            inl fn = rust.one_param_fn fn
            inl x = join x
            $'!x.map(|x| !fn(x))' : option' u
        TypeScript = fun () =>
            if $'!x === undefined' : bool
            then none' ()
            else fn $'!x ' |> fun x => $'!x ' : option' u
        Zig = fun () =>
            if $'(!x == null)' : bool
            then none' ()
            else x |> get' |> fn |> some'
        Lean = fun () =>
            if $'(!x).isNone' : bool
            then none' ()
            else x |> get' |> fn |> some'
    }

### map''

In [ ]:
inl map'' forall t u. (fn : t -> u) (x : option' t) : option' u =
    x |> map fn

### unbox

In [ ]:
inl unbox forall t. (x : option' t) : option t =
    x |> map'' Some |> default_value' None

In [6]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

inl x = Some 3i32
inl y : option i32 = None
inl x' = x |> box |> unbox
inl y' = y |> box |> map id |> unbox
(x', y') |> _assert_eq' (x, y)

.gleam (gleam)
__assert_eq' { expected = (Some(3), None) }
.fsx (fsharp)
__assert_eq' { expected = (Some(3), None) }
.py (cuda)
__assert_eq' { expected = (Some(3), None) }
.rs (rust)
__assert_eq' { expected = (Some(3), None) }
.ts (typescript)
__assert_eq' { expected = (Some(3), None) }

0

### of_obj

In [ ]:
inl of_obj forall t. (x : t) : option' t =
    backend_switch {
        Gleam = fun () =>
            option'_types ()
            $'!x |> option.Some' : option' t
        Lua = fun () =>
            $'!x ' : option' t
        Fsharp = fun () => (x |> $'Option.ofObj') : option' t
        Python = fun () =>
            $'!x ' : option' t
        Rust = fun () => $'Some(!x.clone())' : option' t
        TypeScript = fun () => $'(!x ?? undefined)' : option' t
        Zig = fun () => $'!x ' : option' t
    }

In [7]:
///- --test
///> fsharp
///> cuda
///> typescript
///> cuda

null ()
|> of_obj
|> unbox
|> _assert_eq (None : option string)

.fsx (fsharp)
__assert_eq { expected = None }
.py (cuda)
__assert_eq { expected = None }
.ts (typescript)
__assert_eq { expected = None }
.py (cuda)
__assert_eq { expected = None }

In [8]:
///- --test
///> gleam
///> fsharp
///> cuda
///> rust
///> typescript

""
|> of_obj
|> unbox
|> _assert_eq' (Some "")

.gleam (gleam)
__assert_eq' { expected = Some() }
.fsx (fsharp)
__assert_eq' { expected = Some() }
.py (cuda)
__assert_eq' { expected = Some() }
.rs (rust)
__assert_eq' { expected = Some() }
.ts (typescript)
__assert_eq' { expected = Some() }

0

## rust

### try'

In [ ]:
inl try' forall t. (x : option' t) : t =
    !\\(x, $'"$0?"')

### map'

In [ ]:
inl map' forall t u. (fn : t -> u) (x : option' t) : option' u =
    (!\\(x, $'"true; let _optionm_map_ = $0.map(|x| { //"') : bool) |> ignore
    inl result = fn !\($'"x"')
    (!\\(result, $'"true; $0 })"') : bool) |> ignore
    !\($'"_optionm_map_"')

### unwrap

In [ ]:
inl unwrap forall t. (x : option' t) : t =
    !\\(x, $'"$0.unwrap()"')

### take

In [ ]:
inl take forall t. (x : option' t) : option' t =
    (!\\(x, $'"true; let mut !x = !x"') : bool) |> ignore
    !\\(x, $'"Option::take(&mut $0)"')

### take_ref

In [ ]:
inl take_ref forall t. (x : rust.ref (option' t)) : option' t =
    (!\\(x, $'"true; let mut !x = !x"') : bool) |> ignore
    !\\(x, $'"Option::take(&mut $0)"')

### take_ref_mut

In [ ]:
inl take_ref_mut forall t. (x : rust.ref (rust.mut' (option' t))) : option' t =
    !\\(x, $'"Option::take($0)"')

### cloned

In [ ]:
inl cloned forall t. (x : option' (rust.ref t)) : option' t =
    !\\(x, $'"$0.cloned()"')

### as_ref

In [ ]:
inl as_ref forall t. (x : rust.ref (option' t)) : option' (rust.ref t) =
    !\\(x, $'"$0.as_ref()"')

### as_mut

In [ ]:
inl as_mut forall t. (x : rust.ref (rust.mut' (option' t))) : option' (rust.ref (rust.mut' t)) =
    !\\(x, $'"$0.as_mut()"')

### unwrap_or

In [ ]:
inl unwrap_or forall t. (def : t) (x : option' t) : t =
    !\($'"!x.unwrap_or(!def)"')

### and_then

In [ ]:
inl and_then forall t u. (fn : t -> option' u) (x : option' t) : option' u =
    !\\((x, fn), $'"$0.and_then(|x| $1(x))"')

### rc_upgrade

In [ ]:
inl rc_upgrade forall t. (x : rust.weak_rc t) : option' (rust.rc t) =
    !\\(x, $'"std::rc::Weak::upgrade(&$0)"')

### rc_into_inner

In [9]:
inl rc_into_inner forall t. (x : rust.rc t) : option' t =
    !\\(x, $'"std::rc::Rc::into_inner($0)"')

In [10]:
///- --test
///> rust

rust.new_rc 0i32
|> rc_into_inner
|> unbox
|> _assert_eq' (Some 0i32)

__assert_eq' { expected = Some(0) }

0